# 재하봇 호출어 — TTS 생성 A/B 진단

**목적:** v2 본생성에서 나온 positive 음성이 '재하봇'으로 들리지 않았다
(6개 중 5개 실패, 25초짜리 폭주 포함). 4.5시간을 다시 태우기 전에
**어느 설정이 범인인지** 20개씩 뽑아 가린다.

| 변형 | 지금과 다른 점 | 검증하는 가설 |
|---|---|---|
| **A** | 없음 (기준선) | 지금 설정이 얼마나 나쁜가 |
| **B** | `inference_timesteps` 4,6,8 → **8,10,12** | 속도 벌려고 품질을 깎은 게 원인 |
| **C** | B + 문구를 `재하봇/재하봇아/재하봇이` 3개로 | 비어휘 문구(재하보사·재하부사)가 원인 |
| **D** | C + 프롬프트를 짧게 | 긴 수식 프롬프트가 원인 |

**소요:** 생성 25~30분 + 채점 5분. (데이터를 새로 받아야 하면 +30~60분)

---
### 두 가지 실행 경로

- **기존 런타임에서 돌리는 경우**(본생성 돌리던 노트북이 아직 살아 있음) —
  데이터가 이미 있으므로 아래 4번 셀이 자동으로 건너뛴다. 가장 빠르다.
- **새 런타임**(이 노트북만 새로 열었을 때) — VoxCPM 4.7GB + 배경음 1.1GB 를
  받아야 한다. Drive 캐시가 있으면 그걸 쓰고, 없으면 `setup` 을 돌린다.

**런타임 → 런타임 유형 변경 → GPU** 먼저 확인할 것.

In [ ]:
!nvidia-smi
import sys; print('python', sys.version)

In [ ]:
# ── 경로·토큰 ──────────────────────────────────────────────
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # 배경음 다운로드 rate limit 회피

WORK   = '/content/jaeha_wake'
DATA   = os.path.join(WORK, 'data')
OUTDIR = os.path.join(WORK, 'output')
BACKUP = '/content/drive/MyDrive/jaeha_wake_backup'
DATA_TAR = os.path.join(BACKUP, 'jaeha_wake_data.tar')

for d in (WORK, DATA, OUTDIR, BACKUP):
    os.makedirs(d, exist_ok=True)
os.makedirs(os.path.join(WORK, 'configs'), exist_ok=True)
os.chdir(WORK)
print('작업 폴더:', WORK)

In [ ]:
# ── 설치 (이미 깔려 있으면 건너뜀) ──────────────────────────
import importlib.util

if importlib.util.find_spec('voxcpm') is None:
    !apt-get -qq install -y espeak-ng libsndfile1 ffmpeg sox
    !pip -q install 'livekit-wakeword[train,eval,export,voxcpm]'
    !pip -q install faster-whisper
else:
    print('livekit-wakeword/voxcpm 이미 설치됨 — 건너뜀')
    if importlib.util.find_spec('faster_whisper') is None:
        !pip -q install faster-whisper

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU!')

In [ ]:
# ── 진행이 보이는 실행 헬퍼 (셸매직은 출력이 버퍼링돼 죽었는지 도는지 알 수 없다) ──
import subprocess, sys, time
from IPython.display import clear_output

def run_stream(*args, every=15.0, keep_tail=8):
    p = subprocess.Popen(list(args), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail, n, last = [], 0, 0.0
    for line in p.stdout:
        tail.append(line.rstrip())
        if len(tail) > keep_tail: tail.pop(0)
        n += 1
        if time.time() - last >= every:
            clear_output(wait=True)
            print(f'[{n}줄 진행중] {" ".join(args[:2])}')
            print('\n'.join(tail)); sys.stdout.flush()
            last = time.time()
    p.wait()
    clear_output(wait=True)
    print('\n'.join(tail))
    print(f'\n--- exit code: {p.returncode} (총 {n}줄) ---')
    if p.returncode != 0:
        raise RuntimeError(f'{args[0]} 실패(exit {p.returncode}) — 위 로그 확인')

### 기준선 설정 = 지금 불량 음성을 만든 바로 그 config

변형 A 는 이 값 그대로다. B·C·D 는 여기서 한 가지씩만 바꾼다.

In [ ]:
import yaml

BASE_YAML = r'''
model_name: jaehabot_v2
# ============================================================================
# v2 변경점 (2026-08-05) — 빠른 발화 미인식 수정
#
# 문제(2026-08-02 실기): "천천히 부르면 잘 되는데 빠르게 부르면 인식 안 됨"
# 원인: voice_design_prompts 6개에 **속도 묘사가 하나도 없었다.** 공식 영어 base 에는
#       moderate/slightly faster/quick conversational/slower pace 가 골고루 있었는데
#       한국어로 교체하면서 속도 축을 통째로 빠뜨렸다. 결과적으로 긍정 샘플이 전부
#       비슷한 보통~느린 속도가 되어 빠른 발화가 학습 분포 밖에 놓였다.
# ⚠️ length_scales 는 **Piper 전용 키라 voxcpm 백엔드에선 무시된다.**
#    voxcpm 에서 속도를 바꾸는 유일한 수단은 프롬프트 텍스트뿐이다.
#
# 수정: 음색·나이 축 × 속도 축으로 프롬프트를 재구성(6개 → 16개).
#       빠름 7 / 보통 6 / 느림 3. 빠른 쪽에 가중을 뒀다 — 그게 지금 비어 있는 구간이고,
#       느림은 v1 이 사실상 그 구간이라 이미 잘 되므로 회귀 방지 최소분만 남겼다.
# ============================================================================
# 🔴 '재하봇아' 강화 (2026-08-05): 호격 '~아'는 연음으로 [재하보사] 로 실현된다.
#    voxcpm 이 한국어 연음을 제대로 렌더링한다는 보장이 없어 **발음형을 병기**한다.
#    근거: configs/model_paths.yaml 의 stt.aliases 에 제아부사/재아부사 -> 재하봇 이
#    등록돼 있다 = 실제 사람이 부를 때 그렇게 들린다는 뜻.
#
#    v1 의 5개에서 '재하 봇'·'재하보'·'재하봇아아'(잘 안 쓰는 변형)를 빼고
#    호격형과 그 발음 변형에 집중해 7개로 재구성했다. 문구가 줄면 문구당 샘플이
#    두꺼워진다(3000/7 = 약 428개, v1 은 2000/5 = 400개).
target_phrases:
- 재하봇
- 제하봇        # 첫 음절이 '제'로 들리는 변형 (aliases 의 제아부사 계열과 같은 갈래)
- 재하봇아      # 호격 표기형
- 재하보사      # '재하봇아' 연음 실현형 — TTS 가 연음을 안 살릴 때의 보험
- 재하부사      # 실기 오인식형 (aliases: 제아부사/재아부사)
- 재하봇이      # 주격 표기형
- 재하보시      # '재하봇이' 연음 실현형
tts_backend: voxcpm
# 프롬프트가 6 -> 16 으로 늘어 프롬프트당 샘플 수가 1/3 수준으로 얇아진다.
# 2000 을 유지하면 프롬프트당 125개뿐 → 3000 으로 올려 187개를 확보한다.
# ⚠️ 생성 시간이 비례해 늘어난다(voxcpm 은 순차 생성, 2000개 ≈ 2.5~3시간 → 3000개 ≈ 4~4.5시간).
#    시간이 부족하면 2000 으로 되돌리되, 프롬프트 수를 12개로 줄이는 쪽을 먼저 검토할 것.
n_samples: 3000
n_samples_val: 750
n_background_samples: 3000
n_background_samples_val: 750
tts_batch_size: 256     # ⚠️ voxcpm_backend.py 가 `del batch_size` 로 무시한다(순차 생성). 무해해서 남겨둠.
custom_negative_phrases:
- 자동차
- 해바라기
- 지하철
- 재밌다
- 재미있어
- 재현이
- 로봇
- 보트
- 하마
- 자전거
- 엄마
- 아빠
- 이거 뭐야
- 밥 먹자
- 같이 놀자
- 안녕
- 싫어
noise_scales:
- 0.98
noise_scale_ws:
- 0.98
length_scales:          # ⚠️ Piper 전용 — voxcpm 에선 무시됨. 속도는 프롬프트로만 제어된다.
- 0.75
- 1.0
- 1.25
slerp_weights:
- 0.2
- 0.35
- 0.5
- 0.65
- 0.8
piper_tts:
  checkpoint_relpath: piper/en-us-libritts-high.pt
voxcpm_tts:
  model_id: openbmb/VoxCPM2
  model_cache_relpath: voxcpm/VoxCPM2
  local_model_path: null
  load_denoiser: false
  cfg_values:
  - 1.5
  - 2.0
  - 2.5
  - 3.0
  inference_timesteps_list:   # 기본 [8,10,12] 에서 낮춤 — 생성 속도의 유일한 실질 레버
  - 4
  - 6
  - 8
  # ⚠️ 반드시 voxcpm_tts 아래에 중첩할 것. 최상위에 쓰면 조용히 무시되고 원본 영어 33종이 쓰인다.
  voice_design_prompts:
  # --- 빠름 (7개) — v1 에 없던 구간. 실기 실패의 원인이라 가장 두껍게 ---
  - 빠르게 부르는 한국인 어린 여자아이
  - 급하게 외치듯 부르는 한국인 남자아이
  - 신이 나서 빠르게 말하는 어린아이
  - 다급하게 부르는 어린아이
  - 빠른 말투로 또렷하게 말하는 한국인 성인 여성
  - 빠르게 툭 내뱉듯 부르는 어린아이
  - 숨차게 빠르게 부르는 한국인 아이
  # --- 보통 (6개) — v1 프롬프트를 속도 표현만 붙여 승계 ---
  - 보통 속도로 밝고 높게 말하는 한국인 어린 여자아이
  - 보통 속도로 해맑게 말하는 한국인 남자아이
  - 보통 속도로 또렷하게 말하는 한국인 성인 여성
  - 보통 속도로 차분하게 말하는 한국인 성인 남성
  - 보통 속도로 조금 웅얼거리며 말하는 어린아이
  - 신나서 크게 부르는 한국인 어린아이
  # --- 느림 (3개) — v1 이 사실상 이 구간이라 이미 잘 된다. 회귀 방지 최소분만 유지 ---
  - 아주 천천히 또박또박 부르는 한국인 어린아이
  - 천천히 조심스럽게 부르는 한국인 여자아이
  - 느릿하게 웅얼거리는 어린아이
data_dir: /content/jaeha_wake/data
output_dir: /content/jaeha_wake/output
augmentation:
  clip_duration: 2.0
  batch_size: 64
  rounds: 3
  background_paths:
  - ./data/backgrounds
  rir_paths:
  - ./data/rirs
model:
  model_type: conv_attention
  model_size: medium
steps: 50000
learning_rate: 0.0001
weight_decay: 0.01
label_smoothing: 0.05
max_negative_weight: 3000
target_fp_per_hour: 0.1
batch_n_per_class:
  positive: 50
  adversarial_negative: 50
  ACAV100M_sample: 1024
  background_noise: 50
'''

base = yaml.safe_load(BASE_YAML)
base['data_dir'], base['output_dir'] = DATA, OUTDIR

CFG_PATH = 'configs/base_v2.yaml'
yaml.safe_dump(base, open(CFG_PATH, 'w', encoding='utf-8'),
               allow_unicode=True, sort_keys=False)
print('기준선 저장:', CFG_PATH)
print('문구:', base['target_phrases'])
print('timesteps:', base['voxcpm_tts']['inference_timesteps_list'])
print('프롬프트:', len(base['voxcpm_tts']['voice_design_prompts']), '개')

In [ ]:
# ── 데이터 준비 (있으면 건너뛴다) ───────────────────────────
have_vox = os.path.isdir(os.path.join(DATA, 'voxcpm'))
have_bg  = os.path.isdir(os.path.join(DATA, 'backgrounds'))

if have_vox and have_bg:
    print('데이터 이미 있음 — 다운로드 건너뜀')
elif os.path.exists(DATA_TAR):
    print('Drive 캐시에서 복원:', DATA_TAR)
    run_stream('tar', '-C', WORK, '-xf', DATA_TAR)
else:
    print('데이터 없음 → setup 실행 (VoxCPM 4.7GB + 배경음 등, 수십 분)')
    run_stream('livekit-wakeword', 'setup', '--config', CFG_PATH)

!du -sh {DATA}/* 2>/dev/null | cat

## A/B 생성 (25~30분)

변형마다 `model_name` 이 달라서(`ab_a`~`ab_d`) 서로도, 본 산출물(`jaehabot_v2`)도
덮어쓰지 않는다. 부정 문구·배경음은 판정에 필요 없으므로 최소로 줄여 시간을 아꼈다.

In [ ]:
import copy

N_AB = 20      # 변형당 positive 개수. 늘리면 판정이 정확해지고 시간은 비례해 는다.

SHORT_PROMPTS = [           # 변형 D — v1 스타일에 속도 표현만 얹은 짧은 프롬프트
    '빠르게 말하는 한국인 어린 여자아이',
    '빠르게 말하는 한국인 남자아이',
    '빠르게 말하는 한국인 성인 여성',
    '밝고 높은 목소리의 한국인 어린 여자아이',
    '해맑게 말하는 한국인 남자아이',
    '또렷하게 말하는 한국인 성인 여성',
    '천천히 말하는 한국인 어린아이',
]
P3 = ['재하봇', '재하봇아', '재하봇이']      # 비어휘 문구 제외

def make(tag, timesteps=None, phrases=None, prompts=None):
    c = copy.deepcopy(base)
    c['model_name'] = f'ab_{tag}'
    c['n_samples'], c['n_samples_val'] = N_AB, 4
    c['n_background_samples'], c['n_background_samples_val'] = 4, 4
    c['custom_negative_phrases'] = ['자동차', '엄마']
    if timesteps: c['voxcpm_tts']['inference_timesteps_list'] = timesteps
    if phrases:   c['target_phrases'] = phrases
    if prompts:   c['voxcpm_tts']['voice_design_prompts'] = prompts
    p = f'configs/ab_{tag}.yaml'
    yaml.safe_dump(c, open(p, 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
    return p

VARIANTS = [
    ('a', make('a')),
    ('b', make('b', timesteps=[8, 10, 12])),
    ('c', make('c', timesteps=[8, 10, 12], phrases=P3)),
    ('d', make('d', timesteps=[8, 10, 12], phrases=P3, prompts=SHORT_PROMPTS)),
]

import time
t0 = time.time()
for tag, path in VARIANTS:
    print(f'\n########## 변형 {tag.upper()} ##########')
    run_stream('livekit-wakeword', 'generate', path, every=20, keep_tail=5)
print(f'\n생성 완료 ({(time.time()-t0)/60:.1f}분) — 다음 셀에서 채점')

## 자동 채점

STT(`large-v3-turbo`)로 전사해서 **정말 호출어를 말하는지** 센다.
판정은 봇 런타임과 같은 방식 — 한글을 자모로 쪼개 편집거리를 재고,
**거리가 작을수록 정답**이다(0 = 완전 일치). `app/wake.py` 와 값이 같은지 대조 검증했다.

두 축으로 거른다: **길이 3초 초과 = 폭주**, **자모거리 0.45 초과 = 발음 틀림**.

In [ ]:
import glob, numpy as np, soundfile as sf
from faster_whisper import WhisperModel

MAX_DUR, MAX_RATIO = 3.0, 0.45

_CHO  = list("ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎ")
_JUNG = list("ㅏㅐㅑㅒㅓㅔㅕㅖㅗㅘㅙㅚㅛㅜㅝㅞㅟㅠㅡㅢㅣ")
_JONG = list(" ㄱㄲㄳㄴㄵㄶㄷㄹㄺㄻㄼㄽㄾㄿㅀㅁㅂㅄㅅㅆㅇㅈㅊㅋㅌㅍㅎ")

def decompose(t):
    out = []
    for ch in t:
        i = ord(ch) - 0xAC00
        if 0 <= i <= 0xD7A3 - 0xAC00:
            out += [_CHO[i // (21 * 28)], _JUNG[(i % (21 * 28)) // 28]]
            if i % 28: out.append(_JONG[i % 28])
        else:
            out.append(ch)
    return out

def _ed(a, b):
    if not a or not b: return len(a) + len(b)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def jamo_ratio(x, y):
    jx, jy = decompose(x), decompose(y)
    return _ed(jx, jy) / (max(len(jx), len(jy)) or 1)

def best_ratio(text, words):
    toks = [t.strip(' .,!?~"\'…') for t in (text or '').split()]
    toks = [t for t in toks if t]
    if not toks: return 1.0, ''
    cands = toks + [a + b for a, b in zip(toks, toks[1:])]
    best, hit = 1.0, ''
    for w in words:
        r = min(jamo_ratio(c, w) for c in cands)
        if r < best: best, hit = r, w
    return best, hit

m = WhisperModel('large-v3-turbo', device='cuda', compute_type='float16')

print(f'{"변형":>4} {"통과율":>9} {"길이탈락":>9} {"발음탈락":>9} {"중앙길이":>9}')
detail = {}
for tag in ['a', 'b', 'c', 'd']:
    clips = sorted(glob.glob(os.path.join(OUTDIR, f'ab_{tag}', 'positive_train', 'clip_*.wav')))
    if not clips:
        print(f'{tag.upper():>4}   (클립 없음 — 생성 실패?)'); continue
    words = yaml.safe_load(open(f'configs/ab_{tag}.yaml', encoding='utf-8'))['target_phrases']
    rows, n_long, n_far = [], 0, 0
    for p in clips:
        dur = sf.info(p).duration
        if dur > MAX_DUR:
            rows.append((os.path.basename(p), dur, '', 1.0, False)); n_long += 1; continue
        segs, _ = m.transcribe(p, language='ko', beam_size=5,
                               condition_on_previous_text=False)
        text = ' '.join(s.text.strip() for s in segs).strip()
        r, _hit = best_ratio(text, words)
        ok = r <= MAX_RATIO
        n_far += (not ok)
        rows.append((os.path.basename(p), dur, text, r, ok))
    detail[tag] = rows
    n_ok = sum(1 for x in rows if x[4])
    print(f'{tag.upper():>4} {n_ok/len(rows)*100:8.1f}% {n_long:9d} {n_far:9d} '
          f'{np.median([x[1] for x in rows]):8.2f}s')

best_tag = max(detail, key=lambda t: sum(1 for x in detail[t] if x[4])) if detail else None
if best_tag:
    print(f'\n--- 최고 변형 {best_tag.upper()} 개별 결과 ---')
    for name, dur, text, r, ok in detail[best_tag]:
        print(f'{"OK " if ok else "탈락"} {name} {dur:5.2f}s 거리 {r:4.2f}  {text[:40]}')

In [ ]:
# ── 귀로도 확인 (최고 변형에서 6개) ─────────────────────────
import random
from IPython.display import Audio, display

if best_tag:
    clips = sorted(glob.glob(os.path.join(OUTDIR, f'ab_{best_tag}',
                                          'positive_train', 'clip_*.wav')))
    for c in random.Random(0).sample(clips, min(6, len(clips))):
        y, sr = sf.read(c)
        print(os.path.basename(c), f'{len(y)/sr:.2f}s')
        display(Audio(y, rate=sr))

## 결과 읽는 법

- **통과율 80% 넘는 변형이 있다** → 그 설정으로 본 생성(`n_samples: 3000`)을 다시 돌린다.
  단, 본 생성에서도 augment 전에 같은 채점을 걸어 **탈락본을 빼고** 학습에 넣을 것.
- **B 만 좋아졌다** → 원인은 `inference_timesteps` 를 깎은 것. v1 도 같은 값이었으니
  v1 의 recall 40% 도 이것 때문이었을 가능성이 크다.
- **C·D 까지 가야 좋아졌다** → 비어휘 문구·긴 프롬프트도 같이 원인.
  이 경우 v2 의 문구 확장(재하보사·재하부사·재하보시)은 접는 게 낫다.
- **전부 80% 미만** → voxcpm 의 한국어가 이 용도에 안 맞는 것.
  TTS 를 교체한다(1순위: 지금 봇 목소리인 **Supertonic** — 한국어 품질이 이미 검증돼 있다).

어느 쪽이든 표를 그대로 캡처해서 공유하면 다음 결정을 같이 정할 수 있다.